# Data Science for Business - Recurrent Neural Network (RNN) on Wine Reviews with TensorFlow

*Session 10 · ISLP 10.5 RNNs, esp. 10.5.1 (embeddings, LSTM) · 10.6 When to Use Deep Learning · Lab 10.9.6*

## Initialize notebook
Load required packages. Set up workspace, e.g., set theme for plotting and initialize the random number generator.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn import metrics

import tensorflow as tf
import keras
from keras import layers

In [ ]:
np.random.seed(42)
tf.random.set_seed(42)

Check if we are running on a GPU (an empty list means we train on the CPU).

In [ ]:
tf.config.list_physical_devices('GPU')

## Problem description

The dataset contains 130,000+ wine reviews (Source: https://www.kaggle.com/datasets/zynicide/wine-reviews) written by professional sommeliers. Besides a text, it contains information about the wine, such as the country of origin, the variety, the winery, the price, and the number of points given by the reviewer. The goal of this notebook is to predict the number of points given by the reviewer (the response) based on the text of the review (the predictor). This is a regression problem, so we use the mean absolute error (MAE) as loss, like ISLP Lab 10.9.1.

In this session we use TensorFlow/Keras instead of PyTorch. ISLP 10.9.6 builds the same kind of model in torch; the concepts are identical, only the syntax differs.

## Load data

Load data from CSV file.

In [ ]:
corpus = pd.read_csv("https://raw.githubusercontent.com/olivermueller/ds4b-2026/main/Session_10/winemag-data-130k-v2.csv")
corpus.head()

## Prepare data

### Split data

Split data into three sets: training, validation, and test (ISLP 5.1.1). To speed up training, we only use a random sample of 10,000 reviews for training.

In [ ]:
training = corpus.iloc[0:80000,].sample(n=10000, random_state=42) # to speed up training
validation = corpus.iloc[80000:100000,]
test = corpus.iloc[100000:,]

For each dataset, store predictors (the review text) and response (the points) in separate variables.

In [ ]:
train_corpus_features = training[["description"]]
train_corpus_target = training[["points"]]
val_corpus_features = validation[["description"]]
val_corpus_target = validation[["points"]]
test_corpus_features = test[["description"]]
test_corpus_target = test[["points"]]

### Create TensorFlow datasets

Next, we have to create [TensorFlow `Datasets`](https://www.tensorflow.org/api_docs/python/tf/data/Dataset) from the pandas dataframes. The use of TensorFlow Datasets follows a common pattern:

1. Create a dataset from raw data (e.g., a pandas dataframe, a CSV file, multiple text files).
2. Apply transformations to preprocess the data in the dataset (e.g., tokenize and vectorize the texts).
3. Iterate over the dataset and process its elements. Iteration happens in a streaming fashion, so the full dataset does not need to fit into memory.

Here, we use the `from_tensor_slices` constructor to create datasets from dataframes. With `.batch(128)`, each element of the dataset is a minibatch of 128 reviews, so stochastic gradient descent uses minibatches of 128 (ISLP 10.7.2). Without it, Keras would train on one review at a time, which is extremely slow.

In [ ]:
batch_size = 128

train_ds = tf.data.Dataset.from_tensor_slices((tf.cast(train_corpus_features.values, tf.string),
                                               tf.cast(train_corpus_target.values, tf.int32))).batch(batch_size)

val_ds = tf.data.Dataset.from_tensor_slices((tf.cast(val_corpus_features.values, tf.string),
                                             tf.cast(val_corpus_target.values, tf.int32))).batch(batch_size)

test_ds = tf.data.Dataset.from_tensor_slices((tf.cast(test_corpus_features.values, tf.string),
                                              tf.cast(test_corpus_target.values, tf.int32))).batch(batch_size)

Display some stats and an example from the first minibatch.

In [ ]:
for inputs, targets in train_ds:
    print("inputs.shape:", inputs.shape)
    print("inputs.dtype:", inputs.dtype)
    print("targets.shape:", targets.shape)
    print("targets.dtype:", targets.dtype)
    print("===")
    print("inputs[0]:", inputs[0])
    print("targets[0]:", targets[0])
    break

### Vectorize documents

We will now use [Keras' `TextVectorization`](https://keras.io/api/layers/preprocessing_layers/text/text_vectorization/) layer to transform raw texts into numerical vectors. To do this, we simply map words to integers (`output_mode = 'int'`). To get document representations of a fixed length, we limit the length of documents to 100 tokens (longer documents will be truncated, shorter documents will be padded with zeros). We also set a maximum vocabulary size of 10,000 words, as in ISLP 10.4.

Note a small difference to ISLP 10.5.1: the book keeps the *last* 500 words of each IMDb review and pads *at the front*; `TextVectorization` keeps the *first* 100 tokens and pads *at the end*. Since wine reviews have about 40 words on average and hardly any has more than 100, almost nothing is truncated, and `mask_zero=True` in the embedding layer below tells the model to ignore the padding.

In [ ]:
max_tokens = 10000
max_length = 100

text_vectorization = layers.TextVectorization(
    max_tokens = max_tokens,
    output_mode = "int",
    output_sequence_length = max_length
)

Some aspects of the `TextVectorization` layer (e.g., the size and contents of the vocabulary) have to be fit using training data, which can be done with the `adapt` function (which can only be applied to the predictors (x) of the dataset). `adapt` lowercases the texts, strips punctuation and keeps the 10,000 most frequent words of the **training** data.

In [ ]:
train_ds_features_only = train_ds.map(lambda x, y: x)
text_vectorization.adapt(train_ds_features_only)

Show the vocabulary that our vectorizer knows after being fit to the training data. Index 0 is reserved for padding (`''`) and index 1 for unknown words (`[UNK]`); then follow the words in order of frequency.

In [ ]:
text_vectorization.get_vocabulary()[0:10]

Now, we can apply our adapted `text_vectorization` layer to all three datasets. `cache()` keeps the vectorized minibatches in memory after the first epoch and `prefetch()` prepares the next minibatch while the model is training on the current one.

In [ ]:
vectorized_train_ds = train_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls = tf.data.AUTOTUNE).cache().prefetch(tf.data.AUTOTUNE)

vectorized_val_ds = val_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls = tf.data.AUTOTUNE).cache().prefetch(tf.data.AUTOTUNE)

vectorized_test_ds = test_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls = tf.data.AUTOTUNE).cache().prefetch(tf.data.AUTOTUNE)

Show results for the first minibatch: each review is now a sequence of 100 integers. (We apply `text_vectorization` directly to one raw minibatch here, so that we don't read the cached dataset only partially.)

In [ ]:
for raw_inputs, targets in train_ds.take(1):
    inputs = text_vectorization(raw_inputs)
    print("inputs.shape:", inputs.shape)
    print("inputs.dtype:", inputs.dtype)
    print("targets.shape:", targets.shape)
    print("targets.dtype:", targets.dtype)
    print("===")
    print("inputs[0]:", inputs[0])
    print("targets[0]:", targets[0])

We can use `get_vocabulary()` to get the word behind a specific index.

In [ ]:
text_vectorization.get_vocabulary()[1385]

## Train model

### Network architecture

We are now ready to specify a neural network and feed it with the vectorized datasets. We use Keras' functional API: each layer is applied to the output of the previous one.

- The `Embedding` layer is a trainable lookup table that maps each word index to a vector of length `embedding_dim` (ISLP 10.5.1).
- Instead of averaging or flattening the outputs of the embedding layer, an RNN/LSTM layer can directly process its 2D output (i.e., it takes a sequence of vectors as input instead of a single vector). The LSTM reads the review word by word and keeps a hidden state with 32 hidden units, so word order and negation can matter (ISLP 10.5, 10.5.1), at the price of much slower training. `return_sequences=False` means that only the final hidden state $A_L$ is passed on to the output layer.
- A linear output unit for the predicted points. We initialize its bias with the average points in the training data (about 88). Otherwise, the network would start by predicting values around 0 and would spend many epochs just moving its predictions up to the right range.

ISLP 10.5.1 uses an embedding size of $m = 32$ and notes that the embedding matrix is by far the biggest cost. With `embedding_dim = 300` (the size of popular pretrained embeddings such as GloVe), our embedding layer alone has $10{,}000 \cdot 300 = 3$ million parameters.

In [ ]:
embedding_dim = 300
mean_points = train_corpus_target["points"].mean()  # start the output unit at the average rating

inputs = keras.Input(shape=(max_length,), dtype="int64")
embedded = layers.Embedding(input_dim=max_tokens, output_dim=embedding_dim, mask_zero=True)(inputs)
hidden = layers.LSTM(32, return_sequences=False)(embedded)
outputs = layers.Dense(1, activation="linear", bias_initializer=keras.initializers.Constant(mean_points))(hidden)
model = keras.Model(inputs, outputs)

Compile the model by choosing an optimizer (Adam, a variant of stochastic gradient descent) and a loss function. We minimize the mean absolute error, the same loss as in ISLP Lab 10.9.1.

In [ ]:
model.compile(optimizer = keras.optimizers.Adam(),
              loss = "mean_absolute_error",
              metrics = ["mean_absolute_error"])

Show the model's architecture.

In [ ]:
model.summary()

### Fit the model

Fit the model on the training data. We use the validation set to monitor how the performance of the model evolves during training, and two callbacks:

- `EarlyStopping` stops training when the validation loss has not improved for 2 epochs and restores the weights of the best epoch: early stopping (ISLP 10.7.2) on the validation set, never on the test set.
- `ModelCheckpoint(save_best_only=True)` saves the model of the epoch with the lowest validation loss to the file `lstm.keras`, so we can reload it later without retraining.

The batch size is already set in the datasets, so we don't pass `batch_size` to `fit`. Keras cannot reshuffle a `tf.data.Dataset` itself, so we set `shuffle=False`; our training reviews are already in random order thanks to `.sample()` above.

In [ ]:
callbacks = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True),
             keras.callbacks.ModelCheckpoint("lstm.keras", monitor="val_loss", save_best_only=True)]

history = model.fit(vectorized_train_ds,
                    validation_data = vectorized_val_ds,
                    epochs = 10,
                    shuffle = False,  # order is set by the dataset (see above)
                    callbacks = callbacks)

Plot the learning process.

In [ ]:
plt.plot(history.history['mean_absolute_error'])
plt.plot(history.history['val_mean_absolute_error'])
plt.title('Model MAE')
plt.ylabel('Mean absolute error')
plt.xlabel('Epoch')
plt.legend(['train', 'validation'], loc='upper right')
plt.show()

A validation MAE that rises while the training MAE keeps falling indicates overfitting; this is where early stopping kicks in.

## Make predictions

Load the best model from the training phase.

In [ ]:
model = keras.models.load_model("lstm.keras")

Make predictions on the test set.

In [ ]:
preds = model.predict(vectorized_test_ds)

Calculate the prediction error (MAE) on the test set.

In [ ]:
print(metrics.mean_absolute_error(test_corpus_target, preds))

Is this good? As a naive baseline, we always predict the mean points of the training set, no matter what the review says.

In [ ]:
baseline_preds = np.full(len(test_corpus_target), train_corpus_target["points"].mean())
print(metrics.mean_absolute_error(test_corpus_target, baseline_preds))

The LSTM clearly beats the naive baseline. Compare its test MAE with the MLP from the other notebook. In ISLP 10.5.1, on the IMDb reviews, an LSTM reached 87% test accuracy, comparable with the 88% of the much simpler bag-of-words model (ISLP 10.4), while a simple RNN only reached 76%. Expect something similar here: taking word order into account does not automatically pay off, and the LSTM takes much longer to train (see also ISLP 10.6, *When to Use Deep Learning*).

## Your turn!

**Your turn!** Change the embedding size to 32 (as in ISLP 10.5.1), or replace the `LSTM` layer by a `SimpleRNN` layer (ISLP 10.5). Rebuild, compile and fit the model. How do the number of parameters, the test MAE and the training time change?

In [ ]:
# YOUR CODE HERE